## **Práctica 1**
### **Parte 1.**
---
#### Encargado: **Marcos Bermejo Agenjo**
---


Consiste en la programación de un script que resuelva un problema de programación lineal. Se puede introducir ya en forma matricial sin necesidad de ir capturando una por una las entradas, como si de un software comercial se tratase.

Se trata de programar el algoritmo **«Símplex completo»** visto en clase. Decidirá automáticamente si empezar con Símplex primal o dual y cambiará si fuera necesario. **No hace falta considerar casos degenerados.**

---
##### **Solución:**

#### Entrada del algoritmo

Se considera que el programa recibe un problema de programación lineal en forma matricial, **sin incluir inicialmente las variables de holgura o superávit**.

Por ejemplo, dado el problema:

$$
\max z = 5x_1 + 4x_2
$$

Sujeto a:

$$
\begin{cases}
4x_1 + 2x_2 \leq 8 \\
x_1 + x_2 \leq 3 \\
x_1,x_2 \geq 0
\end{cases}
$$

La entrada se representa mediante:

$$
c = \begin{pmatrix} 5 & 4 \end{pmatrix},
\qquad
A = \begin{pmatrix}
4 & 2 \\
1 & 1
\end{pmatrix},
\qquad
b = \begin{pmatrix} 8 \\ 3 \end{pmatrix}
$$

Además, se especifican el **sentido de optimización** (maximización o minimización) y el **tipo de cada restricción** ($\leq$ o $\geq$).

El algoritmo se encargará de introducir las variables auxiliares necesarias y construir la matriz del Símplex.



#### Estructura de la implementación

Para implementar el algoritmo **Símplex completo**, dividiremos el programa en cuatro funciones principales y una función auxiliar.

##### 1. `preprocesar()`

Recibe el problema de programación lineal en su formulación original:

$$
\begin{aligned}
\operatorname{opt}\quad & z = c^T x \\
\text{sujeto a}\quad & Ax \;(\leq,\geq)\; b \\
& x \geq 0
\end{aligned}
$$

Donde $\operatorname{opt}$ indica maximización o minimización.

Su objetivo es preparar el problema para aplicar el algoritmo:

- Identificar el sentido de optimización.
- Examinar los signos y tipos de las restricciones.
- Introducir las variables de holgura o superávit necesarias.
- Construir la matriz inicial del Símplex.
- Identificar la base inicial y las transformaciones realizadas.

##### 2. `simplex_estandar()`

Implementa el algoritmo Símplex estándar o primal.

Partiendo de una **solución básica factible**, realiza iterativamente:

1. Comprobación de la condición de optimalidad.
2. Selección de la variable de entrada.
3. Selección de la variable de salida.
4. Actualización de la base.
5. Repetición hasta alcanzar una condición de parada.

##### 3. `simplex_dual()`

Implementa el algoritmo Símplex dual.

Partiendo de una **tabla dual factible**, realiza iterativamente:

1. Comprobación de la factibilidad primal.
2. Selección de la variable de salida.
3. Selección de la variable de entrada.
4. Actualización de la base.
5. Repetición hasta alcanzar una condición de parada.

##### 4. `simplex_completo()`

Función principal encargada de coordinar el procedimiento.

1. Llama a `preprocesar()`.
2. Comprueba las condiciones de la matriz inicial.
3. Decide si comenzar mediante Símplex estándar o dual.
4. Ejecuta el método correspondiente.
5. Cambia de método cuando sea necesario y se cumplan sus condiciones de aplicación.
6. Devuelve la solución en términos del problema original.

**Nota:** El procedimiento combinado estudiado en clase no garantiza la resolución de todas las formulaciones posibles, especialmente si la tabla inicial no es primal ni dual factible.

##### 5. `actualizar_matriz()` (auxiliar)

Ambos métodos comparten la operación de actualización de la matriz a partir de la base actual.

Siguiendo la formulación de los apuntes, cada iteración se calcula utilizando la **matriz original**, en lugar de la matriz obtenida en la iteración anterior.

Sea $B$ la matriz formada por las columnas básicas de $A$, y $c_B$ el vector de coeficientes de las variables básicas.

La matriz correspondiente a la base actual se obtiene mediante:

$$
\boxed{
\begin{pmatrix}
1 & c_B^T B^{-1} \\
0 & B^{-1}
\end{pmatrix}
\begin{pmatrix}
1 & -c^T & 0 \\
0 & A & b
\end{pmatrix}
}
$$

Por tanto, esta función será compartida por `simplex_estandar()` y `simplex_dual()`, evitando duplicar cálculos.


#### Esquema general

$$
\boxed{\text{Problema original}}
$$

$$
\downarrow
$$

$$
\boxed{\texttt{preprocesar()}}
$$

$$
\downarrow
$$

$$
\boxed{\texttt{simplex\_completo()}}
$$

$$
\downarrow
$$

$$
\begin{cases}
\texttt{simplex\_estandar()} & \text{si hay factibilidad primal}\\
\texttt{simplex\_dual()} & \text{si hay factibilidad dual}
\end{cases}
$$

Ambos métodos utilizan `actualizar_matriz()`.

El resultado final incluirá la solución, el valor de la función objetivo y el estado de terminación del algoritmo.


In [1]:
import numpy as np

In [ ]:
def preprocesar(opt: str, c: np.ndarray, A: np.ndarray, rest: list[str], b: np.ndarray):
    
    # Comprobamos que la entrada es válida
    if opt not in ("max", "min"):
        raise ValueError("La optimización debe ser 'max' o 'min'.")
    if any(r not in ("<=", ">=") for r in rest):
        raise ValueError("Las restricciones deben ser '<=' o '>='.")
    if len(rest) != len(b) or len(b) != len(A):
        raise ValueError("El número de restricciones no coincide.")
    if len(c) != A.shape[1]:
        raise ValueError("El número de variables no coincide.")
    if A.ndim != 2 or b.ndim != 1 or c.ndim != 1:
        raise ValueError("Dimensiones de la entrada incorrectas.")
    
    n_rest = len(rest)
    n_var = len(c)
    c_, A_, rest, b_ = c.copy(), A.copy(), b.copy()
        
    # Convertimos todas las restricciones a '<='
    for i in range(n_rest):
        if rest[i] == ">=":
            A_[i] *= -1
            b_[i] *= -1
            
    # Añadimos variables de holgura
    I_n = np.eye(n_rest)
    A_ = np.hstack((A_, I_n))
    c_ = np.hstack((c_, np.zeros(n_rest)))
    
    # Calculamos la matriz inicial
    renglon_z = np.hstack(([1], -c_, [0]))
    restricciones = np.hstack((np.zeros((n_rest, 1)),
                               A_,
                               b_.reshape(-1, 1)
                               ))
    M0 = np.vstack((renglon_z, restricciones))
    
    return opt, M0